# धडा १० - उत्पादनातील AI एजंट्स

या धड्यांत तुम्ही Microsoft Agent Framework (Python) वापरून AI एजंट्ससाठी **उत्पादन पॅटर्न** शिकाल. आपण यामध्ये समाविष्ट आहोत:

- **निरीक्षणक्षम्यता** — एजंटच्या संवादांमध्ये वेळ मोजणे आणि लॉगिंग जोडणे
- **मूल्यमापन** — प्रतिसादाच्या गुणवत्तेचे मूल्यमापन करण्यासाठी एक मूल्यमापक एजंट वापरणे
- **खर्च व्यवस्थापन** — टोकन ऑप्टिमायझेशन आणि मॉडेल निवडीसाठी धोरणे

या परिस्थितीत एक **प्रवास एजंट** आहे जो वापरकर्त्यांना सहली योजना बनविण्यास मदत करतो, ज्यावर निरीक्षण आणि मूल्यमापन लागू केले आहे.


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## उत्पादन विचार

एआय एजंट्सना प्रोटोटाइपमधून उत्पादनात नेण्यासाठी तीन स्तंभांकडे काळजीपूर्वक लक्ष देणे आवश्यक आहे:

1. **दृष्यतेची क्षमता** — एजंट काय करत आहे, किती वेळ लागतो आणि कोणती साधने कॉल करतो याची तुम्हाला दृश्यता आवश्यक आहे. ट्रेसिंग आणि लॉगिंग शिवाय उत्पादनातील समस्या डिबग करणे जवळजवळ अशक्य आहे.

2. **मूल्यमापन** — स्वयंचलित गुणवत्तेची तपासणी एजंटच्या प्रतिसादांना वेळोवेळी अचूक, पूर्ण आणि उपयुक्त राहण्यासाठी सुनिश्चित करते. एक मूल्यमापक एजंट निश्चित केलेल्या निकषांनुसार प्रतिसादांना गुण देऊ शकतो.

3. **खर्च व्यवस्थापन** — टोकन वापर हा थेट खर्चावर परिणाम करतो. प्रॉम्प्ट ऑप्टिमायझेशन, मॉडेल निवड आणि कॅशिंग सारख्या धोरणांमुळे खर्च नियंत्रणात ठेवण्यास मदत होते, गुणवत्ता वगळता नाही.


## एक निरीक्षणीय एजंट तयार करणे

आम्ही प्रवास साधने परिभाषित करतो आणि एजंट कॉलला वेळेच्या मोजणीसह वेढून घेतो ज्यामुळे आपण विलंब काळाचे निरीक्षण करू शकतो. उत्पादनात, आपण OpenTelemetry किंवा तत्सम ट्रेसिंग बॅकएंडसह एकत्रित करता.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## मूल्यांकन नमुने

एका सामान्य उत्पादन नमुन्यामध्ये दुसर्या एजंटचा वापर **मूल्यांकन करणारा** म्हणून केला जातो. मूल्यांकन करणारा प्रायमरी एजंटच्या उत्तराचे पूर्वनिर्धारित निकषांच्या आधारे पूर्णता, अचूकता आणि उपयुक्तता यांसारख्या निकषांवर गुणात्मक मूल्यांकन करतो.

हे सक्षम करते:
- वापरकर्त्यांपर्यंत उत्तर पोहोचण्यापूर्वी स्वयंचलित गुणवत्ता दरवाजे
- जेव्हा प्रॉम्प्ट किंवा मॉडेल बदलतात तेव्हा पुनरावृत्ती शोधणे
- वेळोवेळी एजंटच्या कामगिरीचा सातत्याने निरीक्षण


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## खर्च व्यवस्थापन धोरणे

उत्पादन AI एजंटसाठी खर्च नियंत्रण खूप महत्त्वाचे आहे. येथे मुख्य धोरणे दिली आहेत:

| धोरण | वर्णन |
|---|---|
| **प्रॉम्प्ट ऑप्टिमायजेशन** | सिस्टम सूचनांना संक्षिप्त ठेवा. इनपुट टोकन्स कमी करण्यासाठी अनावश्यक संदर्भ काढून टाका. |
    "| **मॉडेल निवड** | वर्गीकरण किंवा एक्स्ट्रॅक्शनसारख्या सोप्या कामांसाठी लहान, स्वस्त मॉडेल (उदाहरणार्थ GPT-4.1-मिनी) वापरा आणि गुंतागुंतीच्या तर्कासाठी मोठ्या मॉडेल राखून ठेवा. |\n",
| **कॅशिंग** | टूल परिणाम आणि वारंवार वापरल्या जाणार्‍या क्वेरीज कॅश करा जेणेकरून पुनरावृत्ती API कॉल टाळता येतील. |
| **टोकन बजेट** | अनपेक्षितपणे लांब प्रतिसाद टाळण्यासाठी `max_tokens` मर्यादा सेट करा. |
| **बॅचिंग** | शक्य असल्यास अनेक वापरकर्ता क्वेरीज एकाच API कॉलमध्ये गटबद्ध करा. |

व्यवहारात, एक स्तरित पध्दत चांगली काम करते: सोप्या विनंत्या वेगवान, कमी खर्चिक मॉडेलकडे पाठवा आणि फक्त गुंतागुंतीच्या क्वेरीज अधिक सक्षम मॉडेलकडे पाठवा.


## सारांश

या धड्यात तुम्ही कसे शिकले:

1. एजंट परस्परसंवादांमध्ये काळजीपूर्वक निरीक्षण (observability) कसे जोडायचे, ज्यामध्ये वेळ मोजणी आणि लॉगिंगचा समावेश आहे, ज्यामुळे ट्रेसिंग आणि मॉनिटरिंगसाठी पाया तयार होतो.
2. स्वतःहून एजंट प्रतिसादांचा मूल्यांकन कसा करायचा, ज्यासाठी एक मूल्यांकन करणारा एजंट वापरला जातो जो पूर्णता, अचूकता आणि उपयुक्तता यांचा गुणांकन करतो.
3. प्रॉम्प्ट ऑप्टिमायझेशन, मॉडेल निवड, कॅशिंग आणि टोकन बजेटसाठी खर्च कसा नियंत्रित करायचा.

हे उत्पादन नमुने तुमच्या AI एजंट्सना विश्वासार्ह, मोजता येण्याजोगे आणि प्रमाणात खर्च-प्रभावी होण्यासाठी मदत करतात.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
